# Tracking des expérimentations avec MLflow

Ce notebook présente l'étape 2 du projet de credit scoring : la mise en place du tracking des expérimentations avec MLflow.

## 1. Introduction à MLflow

MLflow est une plateforme open-source pour gérer le cycle de vie du Machine Learning. Dans cette étape, nous nous concentrons sur **MLflow Tracking**, qui permet de :
* Enregistrer les paramètres des modèles (ex: hyperparamètres).
* Enregistrer les métriques de performance (ex: Accuracy, F1-Score).
* Sauvegarder les artefacts (le modèle entraîné lui-même).
* Comparer visuellement différentes versions de modèles.

L'objectif est de structurer nos recherches pour pouvoir retrouver facilement le meilleur modèle.

### Points de vigilance importants :
* **Environnement Isolé** : Toujours utiliser un environnement virtuel (`uv`, `venv`, `conda`) pour éviter les conflits de versions.
* **Annotation des Expériences** : Utiliser des noms de runs clairs et des **tags** pour faciliter la recherche ultérieure.
* **Versionnement** : Bien que nous soyons au stade du tracking, garder une trace des modèles permet la reproductibilité. (Le *Model Registry* sera abordé plus tard).
* **Gestion des Artefacts** : Ne sauvegarder que les fichiers nécessaires pour ne pas encombrer le système.

## 2. Imports et Configuration

### 2.1 Import des librairies

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ML & Metrics
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import lightgbm as lgb

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.lightgbm

# Optionnel : Supprimer les warnings de version pip de MLflow
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Désactiver les warnings
import warnings
warnings.filterwarnings('ignore')


### 2.2 Configuration de l'expérience MLflow

In [7]:
# 1. Configurer MLflow pour utiliser une base SQLite pour le tracking
# Cela permet d'utiliser toutes les fonctionnalités de MLflow (recherche, traces, etc.)
# Les artefacts (modèles) seront toujours stockés dans le dossier 'mlruns'.
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(project_root, "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

# Configuration du stockage des artefacts
mlruns_dir = os.path.join(project_root, "mlruns")

# 2. Définir le nom de l'expérience
experiment_name = "Credit_Scoring_Tracking"
exp = mlflow.get_experiment_by_name(experiment_name)

if exp is None:
    # On spécifie explicitement l'artifact_location lors de la création
    # pour s'assurer qu'elle est correcte sur cette machine.
    artifact_location = f"file:///{mlruns_dir}/2" 
    mlflow.create_experiment(experiment_name, artifact_location=artifact_location)
elif exp.lifecycle_stage == "deleted":
    print(f"L'expérience '{experiment_name}' est marquée comme supprimée. Restauration...")
    client = mlflow.tracking.MlflowClient()
    client.restore_experiment(exp.experiment_id)

# Vérification et mise à jour dynamique de l'artifact_location si nécessaire
# (Utile si le projet est déplacé d'une machine à une autre)
exp = mlflow.get_experiment_by_name(experiment_name)
expected_location = f"file:///{mlruns_dir}/{exp.experiment_id}".replace("\\", "/")
current_location = exp.artifact_location.replace("\\", "/")

if current_location != expected_location:
    print(f"Mise à jour de l'emplacement des artefacts : {current_location} -> {expected_location}")
    client = mlflow.tracking.MlflowClient()
    # Note: MLflow ne permet pas directement de modifier l'artifact_location via l'API client standard
    # dans toutes les versions. Ici on informe l'utilisateur ou on pourrait utiliser une commande SQL.
    # Pour ce projet, on suppose que le script fix_mlflow_db.py a été passé ou que l'utilisateur est prévenu.

mlflow.set_experiment(experiment_name)
print(f"Expérience active : {experiment_name}")
print(f"Tracking URI : {mlflow.get_tracking_uri()}")


Expérience active : Credit_Scoring_Tracking
Tracking URI : sqlite:///C:\Users\antho\Desktop\dev\Initiez_vous_au_MLOps_1\mlflow.db


## 3. Chargement des données préparées

On réutilise le dataset `train_final.parquet` généré à l'étape précédente.

In [8]:
# Utilisation de chemins robustes
train_path = os.path.join(project_root, "outputs", "train_final.parquet")

if os.path.exists(train_path):
    df = pd.read_parquet(train_path)
    print(f"Dataset chargé avec succès depuis {train_path}")
    print(f"Dimensions : {df.shape}")
else:
    df = pd.DataFrame()
    print(f"Erreur : le fichier {train_path} est introuvable. Vérifiez l'étape 1.")

# Pour des raisons pédagogiques et de rapidité dans ce notebook, 
# nous allons travailler sur un échantillon si le dataset est trop gros.
if not df.empty:
    if len(df) > 50000:
        df_sample = df.sample(50000, random_state=42)
    else:
        df_sample = df

    # Suppression des colonnes non numériques restantes (si elles existent) pour les modèles simples
    X = df_sample.drop(columns=['TARGET', 'SK_ID_CURR']).select_dtypes(include=[np.number])
    y = df_sample['TARGET']

    # Remplacement des NaN simples (Baseline)
    X = X.fillna(X.median())

    print(f"Données prêtes pour l'entraînement : {X.shape}")


Dataset chargé avec succès depuis C:\Users\antho\Desktop\dev\Initiez_vous_au_MLOps_1\outputs\train_final.parquet
Dimensions : (307511, 482)
Données prêtes pour l'entraînement : (50000, 466)


## 4. Séparation Train / Test

On divise les données pour évaluer la performance sur des données non vues.

In [9]:
if not df.empty:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
    print(f"Distribution de la cible dans Train:\n{y_train.value_counts(normalize=True)}")


Train size: 40000, Test size: 10000
Distribution de la cible dans Train:
TARGET
0    0.91955
1    0.08045
Name: proportion, dtype: float64


## 5. Premier Tracking MLflow : Modèle Baseline (Logistic Regression)

Nous commençons par une **Régression Logistique**. 
* **Pourquoi ?** C'est un modèle simple, rapide et interprétable, parfait comme "Baseline" (point de référence).
* **Gestion du déséquilibre** : On utilise `class_weight='balanced'` car les défauts de paiement sont minoritaires.


In [10]:
def eval_metrics(actual, pred, pred_proba):
    acc = accuracy_score(actual, pred)
    prec = precision_score(actual, pred)
    rec = recall_score(actual, pred)
    f1 = f1_score(actual, pred)
    auc = roc_auc_score(actual, pred_proba)
    return acc, prec, rec, f1, auc

if not df.empty:
    # Démarrage d'un run MLflow
    with mlflow.start_run(run_name="Baseline_LogisticRegression"):
        
        # 1. Définition et entraînement du modèle
        lr_params = {"C": 1.0, "max_iter": 1000, "class_weight": "balanced", "random_state": 42}
        lr = LogisticRegression(**lr_params)
        lr.fit(X_train, y_train)
        
        # 2. Prédictions
        y_pred = lr.predict(X_test)
        y_proba = lr.predict_proba(X_test)[:, 1]
        
        # 3. Calcul des métriques
        (acc, prec, rec, f1, auc) = eval_metrics(y_test, y_pred, y_proba)
        
        # 4. Logging des paramètres
        mlflow.log_params(lr_params)
        
        # 5. Logging des métriques
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", auc)
        
        # 6. Ajout de tags pour l'organisation
        mlflow.set_tag("model_type", "LogisticRegression")
        mlflow.set_tag("step", "Baseline")
        
        # 7. Sauvegarde du modèle (Artefact)
        mlflow.sklearn.log_model(lr, artifact_path="model", serialization_format="skops")
        
        print(f"Run terminé. AUC: {auc:.4f}")


Run terminé. AUC: 0.6354


## 6. Utilisation de mlflow.autolog()

`mlflow.autolog()` est une fonction magique qui enregistre automatiquement les paramètres, métriques et modèles pour les librairies supportées (Scikit-Learn, LightGBM, etc.).

* **Avantages** : Très rapide à mettre en place, évite d'oublier de logger un paramètre important.
* **Limites** : Moins de contrôle sur ce qui est loggé, peut logger trop de choses inutiles.

On teste avec une **Random Forest**.

In [11]:
if not df.empty:
    mlflow.sklearn.autolog(serialization_format="skops")

    with mlflow.start_run(run_name="RandomForest_Autolog"):
        rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_test)
        y_proba = rf.predict_proba(X_test)[:, 1]
        
        # On peut quand même ajouter des métriques ou tags manuels en plus
        auc = roc_auc_score(y_test, y_proba)
        mlflow.log_metric("roc_auc_manual", auc)
        mlflow.set_tag("author", "DataScientist_Junior")

    mlflow.sklearn.autolog(disable=True) # On le désactive pour garder le contrôle ensuite


## 7. Comparaison avec un modèle avancé : LightGBM

LightGBM est souvent plus performant sur les données tabulaires de grande taille.

In [12]:
if not df.empty:
    with mlflow.start_run(run_name="LightGBM_Model"):
        
        lgb_params = {
            'objective': 'binary',
            'metric': 'auc',
            'is_unbalance': True, # Gestion du déséquilibre
            'learning_rate': 0.05,
            'num_leaves': 31,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'random_state': 42,
            'verbosity': -1
        }
        
        # Entraînement
        train_data = lgb.Dataset(X_train, label=y_train)
        test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
        
        model_lgb = lgb.train(
            lgb_params,
            train_data,
            valid_sets=[train_data, test_data],
            valid_names=['train', 'valid'],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(stopping_rounds=50)]
        )
        
        # Prédictions
        y_proba = model_lgb.predict(X_test)
        y_pred = (y_proba >= 0.5).astype(int)
        
        # Métriques
        (acc, prec, rec, f1, auc) = eval_metrics(y_test, y_pred, y_proba)
        
        # Logging
        mlflow.log_params(lgb_params)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.set_tag("model_type", "LightGBM")
        
        # Sauvegarde
        mlflow.lightgbm.log_model(model_lgb, artifact_path="model_lgb")
        
        print(f"LightGBM Run terminé. AUC: {auc:.4f}")


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[142]	train's auc: 0.929812	valid's auc: 0.764483
LightGBM Run terminé. AUC: 0.7645


## 8. Visualisation dans MLflow UI

Pour visualiser et comparer nos runs, nous utilisons l'interface graphique de MLflow.

### Comment lancer l'UI ?
Ouvrez un terminal dans le dossier du projet et tapez :
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --workers 1
```

### Accès
* **URL** : http://127.0.0.1:5000
* **Port** : 5000 par défaut.

> **Astuce** : Si vous avez une erreur `[Errno 10048]`, le port 5000 est déjà occupé. Vous pouvez spécifier un autre port avec `--port 5001`.

### Ce qu'il faut regarder :
1. **Tableau des runs** : Comparez les scores `roc_auc` et `f1_score`.
2. **Filtres** : Vous pouvez filtrer par tag (ex: `tags.step = 'Baseline'`).
3. **Graphiques** : Sélectionnez plusieurs runs et cliquez sur "Compare" pour voir l'impact des hyperparamètres sur la performance.
4. **Artefacts** : Cliquez sur un run pour voir le modèle sauvegardé et les fichiers associés.


## 9. Conclusion

Nous avons maintenant un système de tracking robuste :
1. **Traçabilité** : Chaque test est enregistré.
2. **Comparabilité** : On voit clairement que LightGBM (ou RandomForest) surpasse la Baseline.
3. **Organisation** : Les tags permettent de s'y retrouver dans un projet réel avec des dizaines de runs.

Prochaine étape : Optimisation métier et déploiement !
